# Field Strategies Evaluation Demo

Demonstrates the new `FieldEvalStrategy` / `DEFAULT_FIELD_STRATEGIES` API introduced in Phase 1
evaluation hardening (WU-EH1–EH4).  
All scenarios re-use the cached batch-abstract predictions from `batch_abstract_evaluation.ipynb`
— **no new API calls are made**.

## What's new

| Old API | New API |
|---|---|
| `fields=[...]` hand-written list | `field_strategies=DEFAULT_FIELD_STRATEGIES` (registry is the list) |
| `temporal_range` / `referred_dataset` silently included | Both **dropped** from registry (redundant / noisy GT) |
| `enhanced_species_matching=True` global flag | `FieldEvalStrategy(match='enhanced_species', threshold=70)` per-field |
| 14 evaluated fields | 12 evaluated fields (cleaner signal) |

## Scenarios

| Demo | Config | Purpose |
|---|---|---|
| **A** | `DEFAULT_FIELD_STRATEGIES` | All 12 registry fields; compare with old 14-field result |
| **B** | `DEFAULT_FIELD_STRATEGIES` + `fields=["data_type", "species"]` | Two-field focused eval |
| **C** | Custom registry — species exact vs enhanced_species vs fuzzy | Strategy impact on one field |
| **D** | Legacy `enhanced_species_matching=True` (empty `field_strategies`) | Backward compat check |

In [1]:
%load_ext autoreload
%autoreload 2

import ast
import os
from pathlib import Path

import pandas as pd

# Ensure cwd is project root (same convention as batch_abstract_evaluation.ipynb)
_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
os.chdir(PROJECT_ROOT)
print(f"Working directory: {PROJECT_ROOT}")

from llm_metadata.groundtruth_eval import (
    DEFAULT_FIELD_STRATEGIES,
    EvaluationConfig,
    FieldEvalStrategy,
    FuzzyMatchConfig,
    evaluate_indexed,
    macro_f1,
    micro_average,
)
from llm_metadata.schemas.fuster_features import DatasetFeatures, DatasetFeaturesNormalized

# Re-use the detail CSV produced by the prior batch run — no API calls, no openpyxl needed.
DETAIL_CSV = PROJECT_ROOT / "notebooks/results/batch_abstract_evaluation_20260218_145142/abstract_evaluation_detail.csv"

print(f"Detail CSV: {DETAIL_CSV.exists()}")
print(f"\nDEFAULT_FIELD_STRATEGIES ({len(DEFAULT_FIELD_STRATEGIES)} fields):")
for fname, strat in DEFAULT_FIELD_STRATEGIES.items():
    print(f"  {fname:<30} match={strat.match!r}  threshold={strat.threshold}")

Working directory: /home/user/llm_metadata


Detail CSV: True

DEFAULT_FIELD_STRATEGIES (12 fields):
  temp_range_i                   match='exact'  threshold=80
  temp_range_f                   match='exact'  threshold=80
  spatial_range_km2              match='exact'  threshold=80
  data_type                      match='exact'  threshold=80
  geospatial_info_dataset        match='exact'  threshold=80
  species                        match='enhanced_species'  threshold=70
  time_series                    match='exact'  threshold=80
  multispecies                   match='exact'  threshold=80
  threatened_species             match='exact'  threshold=80
  new_species_science            match='exact'  threshold=80
  new_species_region             match='exact'  threshold=80
  bias_north_south               match='exact'  threshold=80


## Load Ground Truth and Predictions from Detail CSV

The detail CSV from the prior batch run (`batch_abstract_evaluation_20260218_145142`) stores
`true_value` and `pred_value` per `(record_id, field)` as Python repr strings.

We pivot each column separately:
- **GT** (`true_value`) → `DatasetFeaturesNormalized` (Pydantic validators apply vocab normalization)
- **Predictions** (`pred_value`) → `DatasetFeatures`

No xlsx file or API calls needed.

In [2]:
def _parse_csv_value(s) -> object:
    """Parse a Python repr string from the detail CSV back to a native type."""
    if s is None or (isinstance(s, float) and pd.isna(s)) or str(s).strip() in ('', 'None'):
        return None
    try:
        return ast.literal_eval(str(s).strip())
    except (ValueError, SyntaxError):
        return str(s).strip()


detail_df = pd.read_csv(DETAIL_CSV, dtype={'record_id': str})
print(f"Detail CSV rows: {len(detail_df)}")
print(f"Fields in CSV: {sorted(detail_df['field'].unique())}")
print(f"Records: {detail_df['record_id'].nunique()}")

# ── Ground truth: pivot true_value ────────────────────────────────────────
gt_pivot = detail_df.pivot(index='record_id', columns='field', values='true_value')

ground_truth_by_id: dict[str, DatasetFeaturesNormalized] = {}
gt_errors = []
for record_id, row in gt_pivot.iterrows():
    gt_dict = {}
    for fname in DEFAULT_FIELD_STRATEGIES:
        if fname in row.index:
            gt_dict[fname] = _parse_csv_value(row[fname])
    try:
        ground_truth_by_id[str(record_id)] = DatasetFeaturesNormalized.model_validate(gt_dict)
    except Exception as e:
        gt_errors.append({'id': record_id, 'error': str(e)})

# ── Predictions: pivot pred_value ─────────────────────────────────────────
pred_pivot = detail_df.pivot(index='record_id', columns='field', values='pred_value')

predictions_by_id: dict[str, DatasetFeatures] = {}
pred_errors = []
for record_id, row in pred_pivot.iterrows():
    pred_dict = {}
    for fname in DEFAULT_FIELD_STRATEGIES:
        if fname in row.index:
            pred_dict[fname] = _parse_csv_value(row[fname])
    try:
        predictions_by_id[str(record_id)] = DatasetFeatures.model_validate(pred_dict)
    except Exception as e:
        pred_errors.append({'id': record_id, 'error': str(e)})

common_ids = set(ground_truth_by_id) & set(predictions_by_id)
true_by_id = {rid: ground_truth_by_id[rid] for rid in common_ids}
pred_by_id = {rid: predictions_by_id[rid]  for rid in common_ids}

print(f"\nGround truth loaded:       {len(ground_truth_by_id)} ({len(gt_errors)} errors)")
print(f"Predictions loaded:        {len(predictions_by_id)} ({len(pred_errors)} errors)")
print(f"Common IDs for evaluation: {len(common_ids)}")

Detail CSV rows: 4032
Fields in CSV: ['bias_north_south', 'data_type', 'geospatial_info_dataset', 'multispecies', 'new_species_region', 'new_species_science', 'referred_dataset', 'spatial_range_km2', 'species', 'temp_range_f', 'temp_range_i', 'temporal_range', 'threatened_species', 'time_series']
Records: 288

Ground truth loaded:       288 (0 errors)
Predictions loaded:        288 (0 errors)
Common IDs for evaluation: 288


---
## Demo A — Full Registry: `DEFAULT_FIELD_STRATEGIES`

Pass `field_strategies=DEFAULT_FIELD_STRATEGIES` instead of a hand-written `fields=` list.  
The registry defines **which** fields are evaluated and **how** each is matched.
No more `temporal_range` or `referred_dataset` dragging down aggregate metrics.

In [3]:
config_registry = EvaluationConfig(
    field_strategies=DEFAULT_FIELD_STRATEGIES,
    # Global normalization (unchanged defaults)
    casefold_strings=True,
    strip_strings=True,
    collapse_whitespace=True,
    treat_lists_as_sets=True,
)

report_registry = evaluate_indexed(
    true_by_id=true_by_id,
    pred_by_id=pred_by_id,
    config=config_registry,
    # NOTE: no fields= argument — registry keys ARE the field list
)

print(f"Fields evaluated: {sorted(report_registry.field_metrics.keys())}")
print(f"Count: {len(report_registry.field_metrics)}  (expected 12)\n")

metrics_df = report_registry.metrics_to_pandas()[['field', 'precision', 'recall', 'f1', 'tp', 'fp', 'fn', 'n']]
print("Per-field metrics (DEFAULT_FIELD_STRATEGIES, 12 fields):")
display(metrics_df.sort_values('f1', ascending=False).reset_index(drop=True))

micro = micro_average(report_registry.field_metrics.values())
mf1   = macro_f1(report_registry.field_metrics.values())
print(f"\nMicro F1 (12 fields): {micro.f1:.3f}")
print(f"Macro F1 (12 fields): {mf1:.3f}")

Fields evaluated: ['bias_north_south', 'data_type', 'geospatial_info_dataset', 'multispecies', 'new_species_region', 'new_species_science', 'spatial_range_km2', 'species', 'temp_range_f', 'temp_range_i', 'threatened_species', 'time_series']
Count: 12  (expected 12)

Per-field metrics (DEFAULT_FIELD_STRATEGIES, 12 fields):


,field,precision,recall,f1,tp,fp,fn,n
0,temp_range_i,0.481928,0.357143,0.410256,40,43,72,288
1,temp_range_f,0.451220,0.330357,0.381443,37,45,75,288
2,multispecies,0.292929,0.364780,0.324930,58,140,101,288
3,species,0.162876,0.600000,0.256203,222,1141,148,288
4,time_series,0.112245,0.916667,0.200000,11,87,1,288
5,data_type,0.126888,0.400000,0.192661,84,578,126,288
6,threatened_species,0.473684,0.088235,0.148760,9,10,93,288
7,geospatial_info_dataset,0.064234,0.400000,0.110692,44,641,66,288
8,spatial_range_km2,0.545455,0.057143,0.103448,6,5,99,288
9,new_species_region,0.272727,0.057143,0.094488,6,16,99,288



Micro F1 (12 fields): 0.212
Macro F1 (12 fields): 0.205


In [4]:
# Compare: old 14-field approach (manual fields list, legacy config) vs new registry

OLD_EVAL_FIELDS = [
    'data_type', 'geospatial_info_dataset', 'spatial_range_km2',
    'temporal_range', 'temp_range_i', 'temp_range_f',
    'species', 'referred_dataset',
    'time_series', 'multispecies', 'threatened_species',
    'new_species_science', 'new_species_region', 'bias_north_south',
]

config_old = EvaluationConfig(
    enhanced_species_matching=True,
    enhanced_species_threshold=70,
)

report_old = evaluate_indexed(
    true_by_id=true_by_id,
    pred_by_id=pred_by_id,
    fields=OLD_EVAL_FIELDS,
    config=config_old,
)

micro_old = micro_average(report_old.field_metrics.values())
mf1_old   = macro_f1(report_old.field_metrics.values())

dropped_fields = set(OLD_EVAL_FIELDS) - set(DEFAULT_FIELD_STRATEGIES)
print("Comparison: old 14-field API vs new registry")
print("=" * 50)
print(f"Old fields ({len(OLD_EVAL_FIELDS)}): {sorted(OLD_EVAL_FIELDS)}")
print(f"New fields ({len(DEFAULT_FIELD_STRATEGIES)}): {sorted(DEFAULT_FIELD_STRATEGIES)}")
print(f"Dropped:  {sorted(dropped_fields)}")
print()
print(f"{'Metric':<30} {'Old':>10} {'New':>10}")
print("-" * 52)
print(f"{'Micro F1':<30} {micro_old.f1 or 0:>10.3f} {micro.f1 or 0:>10.3f}")
print(f"{'Macro F1':<30} {mf1_old or 0:>10.3f} {mf1 or 0:>10.3f}")
print(f"{'Fields evaluated':<30} {len(OLD_EVAL_FIELDS):>10} {len(DEFAULT_FIELD_STRATEGIES):>10}")

Comparison: old 14-field API vs new registry
Old fields (14): ['bias_north_south', 'data_type', 'geospatial_info_dataset', 'multispecies', 'new_species_region', 'new_species_science', 'referred_dataset', 'spatial_range_km2', 'species', 'temp_range_f', 'temp_range_i', 'temporal_range', 'threatened_species', 'time_series']
New fields (12): ['bias_north_south', 'data_type', 'geospatial_info_dataset', 'multispecies', 'new_species_region', 'new_species_science', 'spatial_range_km2', 'species', 'temp_range_f', 'temp_range_i', 'threatened_species', 'time_series']
Dropped:  ['referred_dataset', 'temporal_range']

Metric                                Old        New
----------------------------------------------------
Micro F1                            0.212      0.212
Macro F1                            0.205      0.205
Fields evaluated                       14         12


---
## Demo B — Two-Field Focus: `data_type` and `species`

Pass `fields=["data_type", "species"]` alongside `field_strategies=DEFAULT_FIELD_STRATEGIES`.
The `fields=` parameter takes the **intersection** with the registry — only the named subset is evaluated,
using each field's declared strategy (exact for `data_type`, enhanced_species for `species`).

Use case: fast iteration when you've just modified the prompt for these two fields.

In [5]:
report_focused = evaluate_indexed(
    true_by_id=true_by_id,
    pred_by_id=pred_by_id,
    fields=["data_type", "species"],          # intersection with registry
    config=config_registry,
)

print(f"Fields evaluated: {sorted(report_focused.field_metrics.keys())}")
print()

for fname in ['data_type', 'species']:
    m = report_focused.metrics_for(fname)
    print(f"{fname}")
    print(f"  Precision: {m.precision:.3f}  Recall: {m.recall:.3f}  F1: {m.f1:.3f}")
    print(f"  TP={m.tp}  FP={m.fp}  FN={m.fn}  n={m.n}")

Fields evaluated: ['data_type', 'species']

data_type
  Precision: 0.127  Recall: 0.400  F1: 0.193
  TP=84  FP=578  FN=126  n=288
species
  Precision: 0.163  Recall: 0.600  F1: 0.256
  TP=222  FP=1141  FN=148  n=288


In [6]:
# Inspect mismatches for the two-field focused eval
results_df = report_focused.to_pandas()
mismatches = results_df[~results_df['match']].copy()

for fname in ['data_type', 'species']:
    field_mm = mismatches[mismatches['field'] == fname]
    print(f"\n── {fname}: {len(field_mm)} mismatches (showing 5) ──")
    display(
        field_mm[['record_id', 'true_value', 'pred_value', 'tp', 'fp', 'fn']]
        .head(5)
        .reset_index(drop=True)
    )


── data_type: 273 mismatches (showing 5) ──


,record_id,true_value,pred_value,tp,fp,fn
0,10,[presence-absence],"[presence-absence, ecosystem_structure]",1,1,0
1,100,[density],"[distribution, time_series]",0,2,1
2,101,[other],[time_series],0,1,1
3,104,[distribution],"[distribution, ecosystem_structure]",1,1,0
4,105,"[genetic_analysis, other]","[traits, genetic_analysis, time_series]",1,2,1



── species: 244 mismatches (showing 5) ──


,record_id,true_value,pred_value,tp,fp,fn
0,10,[white-tailed deer],"[white-tailed deer, Cornus canadensis, Maianth...",1,9,0
1,103,[Tachycineta bicolor],"[tree swallows, Tachycineta bicolor]",1,1,0
2,105,[Tachycineta bicolor],"[Tachycineta bicolor, tree swallow]",1,1,0
3,107,"[Porzana carolina, Rallus limicola, Coturnicop...","[Sora (Porzana carolina), Porzana carolina, Vi...",3,1,0
4,109,[Salvelinus namaycush],"[stocked, local and hybrid Lake Trout (Salveli...",1,1,0


---
## Demo C — Strategy Impact: Comparing Matching Algorithms for `species`

Swap the matching algorithm for `species` while keeping everything else the same.  
This directly shows how `FieldEvalStrategy` enables controlled, per-field experiment control.

| Config | `species` strategy | Expected behaviour |
|---|---|---|
| `exact` | Exact normalized match | Strictest — penalises "caribou" vs "Rangifer tarandus" |
| `enhanced_species` (default) | Vernacular ↔ scientific name resolution + fuzzy | Handles synonyms and parenthetical forms |
| `fuzzy` (threshold=70) | Fuzzy string similarity on raw strings | Catches typos, no semantic awareness |

In [7]:
def _make_config_with_species(strategy: FieldEvalStrategy) -> EvaluationConfig:
    strategies = dict(DEFAULT_FIELD_STRATEGIES)   # shallow copy
    strategies["species"] = strategy
    return EvaluationConfig(field_strategies=strategies)


configs = {
    "exact":            _make_config_with_species(FieldEvalStrategy(match="exact")),
    "enhanced_species": config_registry,                                                 # default
    "fuzzy (t=70)":     _make_config_with_species(FieldEvalStrategy(match="fuzzy", threshold=70)),
    "fuzzy (t=85)":     _make_config_with_species(FieldEvalStrategy(match="fuzzy", threshold=85)),
}

rows = []
for label, cfg in configs.items():
    rpt = evaluate_indexed(
        true_by_id=true_by_id,
        pred_by_id=pred_by_id,
        fields=["species"],
        config=cfg,
    )
    m = rpt.metrics_for("species")
    rows.append({
        "strategy":  label,
        "precision": round(m.precision or 0, 3),
        "recall":    round(m.recall    or 0, 3),
        "f1":        round(m.f1        or 0, 3),
        "tp": m.tp, "fp": m.fp, "fn": m.fn,
    })

cmp_df = pd.DataFrame(rows).set_index("strategy")
print("Species matching strategy comparison (all 288 records):")
display(cmp_df)

Species matching strategy comparison (all 288 records):


,precision,recall,f1,tp,fp,fn
strategy,,,,,,
exact,0.073,0.273,0.115,101,1280,269
enhanced_species,0.163,0.600,0.256,222,1141,148
fuzzy (t=70),0.127,0.470,0.200,174,1196,196
fuzzy (t=85),0.086,0.319,0.135,118,1262,252


---
## Demo D — Backward Compatibility

The old `EvaluationConfig` parameters (`enhanced_species_matching`, `fuzzy_match_fields`) still work
unchanged when `field_strategies` is empty (`{}`).  
This confirms no regressions for existing notebooks.

In [8]:
# Legacy config — no field_strategies
config_legacy = EvaluationConfig(
    field_strategies={},                    # empty → legacy path
    enhanced_species_matching=True,
    enhanced_species_threshold=70,
)

report_legacy = evaluate_indexed(
    true_by_id=true_by_id,
    pred_by_id=pred_by_id,
    fields=["species", "data_type", "temp_range_i"],
    config=config_legacy,
)

# New config — equivalent behaviour via field_strategies
config_new_equiv = EvaluationConfig(
    field_strategies={
        "species":     FieldEvalStrategy(match="enhanced_species", threshold=70),
        "data_type":   FieldEvalStrategy(match="exact"),
        "temp_range_i": FieldEvalStrategy(match="exact"),
    },
)

report_new_equiv = evaluate_indexed(
    true_by_id=true_by_id,
    pred_by_id=pred_by_id,
    config=config_new_equiv,
)

print("Legacy vs new-equivalent config — must produce identical metrics:")
print(f"{'Field':<20} {'Legacy F1':>12} {'New F1':>12} {'Match?':>8}")
print("-" * 56)
all_match = True
for fname in ['species', 'data_type', 'temp_range_i']:
    f1_legacy = report_legacy.metrics_for(fname).f1
    f1_new    = report_new_equiv.metrics_for(fname).f1
    ok = abs((f1_legacy or 0) - (f1_new or 0)) < 1e-9
    all_match = all_match and ok
    print(f"{fname:<20} {f1_legacy or 0:>12.4f} {f1_new or 0:>12.4f} {'✓' if ok else '✗':>8}")

print()
print(f"All match: {'YES ✓' if all_match else 'NO — REGRESSION DETECTED'}")

Legacy vs new-equivalent config — must produce identical metrics:
Field                   Legacy F1       New F1   Match?
--------------------------------------------------------
species                    0.2562       0.2562        ✓
data_type                  0.1927       0.1927        ✓
temp_range_i               0.4103       0.4103        ✓

All match: YES ✓


---
## Summary

The table below collects the headline numbers from each demo.

In [9]:
micro_reg = micro_average(report_registry.field_metrics.values())
mf1_reg   = macro_f1(report_registry.field_metrics.values())
micro_old = micro_average(report_old.field_metrics.values())
mf1_old2  = macro_f1(report_old.field_metrics.values())

summary_rows = [
    {"Demo": "Old API (14 fields, enhanced_species flag)",
     "Fields": 14,
     "Micro F1": round(micro_old.f1 or 0, 3),
     "Macro F1": round(mf1_old2 or 0, 3),
     "Note": "temporal_range + referred_dataset included"},

    {"Demo": "A — DEFAULT_FIELD_STRATEGIES (12 fields)",
     "Fields": 12,
     "Micro F1": round(micro_reg.f1 or 0, 3),
     "Macro F1": round(mf1_reg or 0, 3),
     "Note": "dropped temporal_range, referred_dataset"},

    {"Demo": "B — data_type + species only",
     "Fields": 2,
     "Micro F1": round(micro_average(report_focused.field_metrics.values()).f1 or 0, 3),
     "Macro F1": round(macro_f1(report_focused.field_metrics.values()) or 0, 3),
     "Note": "fields= intersection with registry"},
]

# Add species strategy comparison rows
for label, cfg in configs.items():
    rpt = evaluate_indexed(
        true_by_id=true_by_id, pred_by_id=pred_by_id,
        fields=["species"], config=cfg,
    )
    m = rpt.metrics_for("species")
    summary_rows.append({
        "Demo": f"C — species({label})",
        "Fields": 1,
        "Micro F1": round(m.f1 or 0, 3),
        "Macro F1": round(m.f1 or 0, 3),
        "Note": f"species-only eval, strategy={label}",
    })

display(pd.DataFrame(summary_rows))

,Demo,Fields,Micro F1,Macro F1,Note
0,"Old API (14 fields, enhanced_species flag)",14,0.212,0.205,temporal_range + referred_dataset included
1,A — DEFAULT_FIELD_STRATEGIES (12 fields),12,0.212,0.205,"dropped temporal_range, referred_dataset"
2,B — data_type + species only,2,0.235,0.224,fields= intersection with registry
3,C — species(exact),1,0.115,0.115,"species-only eval, strategy=exact"
4,C — species(enhanced_species),1,0.256,0.256,"species-only eval, strategy=enhanced_species"
5,C — species(fuzzy (t=70)),1,0.200,0.200,"species-only eval, strategy=fuzzy (t=70)"
6,C — species(fuzzy (t=85)),1,0.135,0.135,"species-only eval, strategy=fuzzy (t=85)"
